# Experimentos — Hackathón PNUD (riesgo de desplazamiento)

Entrena **todas las variantes** con la misma preparación de datos e historial del notebook del equipo, las valida en
los tres cortes temporales con las métricas oficiales y genera **un CSV de entrega por variante**, validado con la
función de la plantilla.

| Variante | Qué cambia respecto al control (`E1_arboles_400`) |
|---|---|
| `V1_early_stopping` | Es la versión 1 ya entregada (early stopping), para comparar |
| `E1_arboles_200` / `E1_arboles_400` / `E1_arboles_800` | Número fijo de árboles del clasificador (200 / 400 / 800); magnitud y cuantiles con 300 |
| `E2_hurdle` | Magnitud en dos partes: regresor entrenado solo con meses con evento × probabilidad |
| `E3_nivel_nacional` | Agrega el nivel nacional reciente de eventos (tendencia de reporte) |
| `E4_min_hoja_200` | Más regularización: mínimo 200 filas por hoja |
| `E4_hojas_15` | Árboles más simples: 15 hojas |
| `COMBO` | E2 + E3 + E4 (mínimo 200 por hoja) |

**Cómo decidir:** la tabla final ordena por el puntaje de validación promedio (F1–F3). Una variante "cumple la regla"
si mejora ≥ 0.5 puntos a la versión 1 sin empeorar en F3. La tabla en vivo **no** decide: solo confirma.
Recuerden: máximo **una entrega cada 30 minutos** y cuenta la **última válida**.

## 1. Datos, calidad y tratamiento (igual que el notebook del equipo)

In [1]:
import os, zipfile
import numpy as np
import pandas as pd

URL_DATOS = "PEGUE_AQUI_EL_ENLACE_DE_LOS_DATOS"  # enlace de Google Drive al archivo hackathon_datos.zip (se comparte el día del evento)
DATA_DIR = os.environ.get("HACKATHON_DATA", "datos")

if not os.path.exists(os.path.join(DATA_DIR, "entrenamiento.csv")):
    if URL_DATOS:
        import gdown
        import re
        ident = re.search(r"/d/([\w-]+)", URL_DATOS)   # enlace directo: funciona con cualquier versión de gdown
        url = f"https://drive.google.com/uc?id={ident.group(1)}" if ident else URL_DATOS
        gdown.download(url, "hackathon_datos.zip", quiet=False)
    with zipfile.ZipFile("hackathon_datos.zip") as z:
        z.extractall(DATA_DIR)

leer = lambda f: pd.read_csv(os.path.join(DATA_DIR, f), dtype={"DIVIPOLA": str}, parse_dates=["fecha"])
train = leer("entrenamiento.csv")
test_pub = leer("prueba_equipos.csv")     # periodo de la tabla de posiciones en vivo
test_priv = leer("prueba_oculta.csv")     # periodo de la nota oficial
dic = pd.read_csv(os.path.join(DATA_DIR, "diccionario_datos.csv"))
print(train.shape, test_pub.shape, test_priv.shape)

Downloading...
From: https://drive.google.com/uc?id=<ID_OCULTO>
To: g:\My Drive\Documentos Juan Felipe\Hackaton_PENUD\notebooks\hackathon_datos.zip
100%|██████████| 959k/959k [00:00<00:00, 6.83MB/s]


(62814, 25) (20938, 13) (22040, 13)


In [2]:
ESPERADAS = {"entrenamiento": 62_814, "prueba_equipos": 20_938, "prueba_oculta": 22_040}
for nombre, df in {"entrenamiento": train, "prueba_equipos": test_pub, "prueba_oculta": test_priv}.items():
    print(f"{nombre:15s} filas={len(df):,} (esperadas {ESPERADAS[nombre]:,}) · municipios={df.DIVIPOLA.nunique()} · "
          f"llaves repetidas={df.duplicated(['DIVIPOLA', 'fecha']).sum()} · celdas nulas={int(df.isna().sum().sum())}")
nulos = pd.concat({"entrenamiento": train.isna().sum(), "prueba_equipos": test_pub.isna().sum(),
                   "prueba_oculta": test_priv.isna().sum()}, axis=1)
display(nulos[nulos.sum(axis=1) > 0])
fuera = pd.concat([test_pub, test_priv]).ONI.gt(train.ONI.max()).mean()
print(f"ONI de prueba por encima del máximo del entrenamiento: {fuera:.1%} de las filas (los árboles no extrapolan)")

entrenamiento   filas=62,814 (esperadas 62,814) · municipios=1102 · llaves repetidas=0 · celdas nulas=162
prueba_equipos  filas=20,938 (esperadas 20,938) · municipios=1102 · llaves repetidas=0 · celdas nulas=38
prueba_oculta   filas=22,040 (esperadas 22,040) · municipios=1102 · llaves repetidas=0 · celdas nulas=40


,entrenamiento,prueba_equipos,prueba_oculta
densidad_poblacional,12,0.0,0.0
ingresos_tributarios,36,0.0,0.0
inversion_gestion_riesgo,57,19.0,20.0
nbi_pct,57,19.0,20.0


ONI de prueba por encima del máximo del entrenamiento: 20.5% de las filas (los árboles no extrapolan)


In [3]:
COV_ESTATICAS = ["nbi_pct", "elevacion_media", "pendiente_media", "pct_pendiente_alta"]
COV_ANUALES = ["densidad_poblacional", "ingresos_tributarios", "inversion_gestion_riesgo"]
RANGOS = {"densidad_poblacional": (0, None), "ingresos_tributarios": (0, None), "inversion_gestion_riesgo": (0, None),
          "nbi_pct": (0, 100), "elevacion_media": (-50, 6000), "pendiente_media": (0, 90),
          "pct_pendiente_alta": (0, 100), "ONI": (-3.5, 3.5)}
HISTORIAL = ["n_eventos", "familias_desplazadas", "viv_destruidas", "viv_averiadas", "n_movimiento_masa",
             "n_inundacion", "n_vendaval", "n_creciente_subita", "n_avenida_torrencial", "n_otros"]
OBJETIVO = ["tiene_evento", "personas_desplazadas"]


def deduplicar(df):
    df = df.drop_duplicates()
    df = (df.assign(_nulos=df.isna().sum(axis=1)).sort_values(["DIVIPOLA", "fecha", "_nulos"])
            .drop_duplicates(["DIVIPOLA", "fecha"]).drop(columns="_nulos"))
    return df.sort_values(["DIVIPOLA", "fecha"]).reset_index(drop=True)


def preparar(train, test_pub, test_priv):
    partes = {}
    for k, df in {"train": train, "pub": test_pub, "priv": test_priv}.items():
        df = df.copy()
        df["DIVIPOLA"] = df["DIVIPOLA"].astype(str).str.strip().str.zfill(5)
        df["anio"], df["mes"] = df["fecha"].dt.year, df["fecha"].dt.month
        n0 = len(df)
        partes[k] = deduplicar(df)
        if len(partes[k]) != n0:
            print(f"{k}: {n0 - len(partes[k])} filas duplicadas eliminadas")

    tr = partes["train"]
    tr["personas_desplazadas"] = tr["personas_desplazadas"].clip(lower=0)
    tr[HISTORIAL] = tr[HISTORIAL].clip(lower=0).fillna(0)
    viv = tr[["viv_destruidas", "viv_averiadas"]].sum(axis=1)
    afectacion = (tr["personas_desplazadas"].fillna(0) > 0) | (viv > 0)
    corregidas = int(((tr["tiene_evento"] != 1) & afectacion).sum())
    tr["tiene_evento"] = np.where(afectacion, 1, tr["tiene_evento"].fillna(0)).astype(int)
    tr["personas_desplazadas"] = tr["personas_desplazadas"].fillna(0)
    print(f"Objetivo corregido según la definición oficial: {corregidas} filas")

    panel = pd.concat([d.assign(_parte=k, _i=np.arange(len(d))) for k, d in partes.items()], ignore_index=True)
    for c, (lo, hi) in RANGOS.items():
        malo = pd.Series(False, index=panel.index)
        if lo is not None:
            malo |= panel[c] < lo
        if hi is not None:
            malo |= panel[c] > hi
        if malo.any():
            print(f"{c}: {int(malo.sum())} valores imposibles -> nulo")
        panel.loc[malo, c] = np.nan
    faltaba = {c: panel[c].isna() for c in RANGOS if panel[c].isna().any()}

    panel["ONI"] = panel["ONI"].fillna(panel.groupby("fecha")["ONI"].transform("median"))
    for c in COV_ESTATICAS:
        panel[c] = panel[c].fillna(panel.groupby("DIVIPOLA")[c].transform("median"))
    for c in COV_ANUALES:   # el panel está en orden cronológico dentro de cada municipio
        panel[c] = panel[c].fillna(panel.groupby(["DIVIPOLA", "anio"])[c].transform("median"))
        panel[c] = panel.groupby("DIVIPOLA")[c].transform(lambda s: s.ffill().bfill())
    dep = panel["DIVIPOLA"].str[:2]
    for c in COV_ESTATICAS + COV_ANUALES:
        grupos = [dep, panel["anio"]] if c in COV_ANUALES else [dep]
        panel[c] = panel[c].fillna(panel.groupby(grupos)[c].transform("median")).fillna(panel[c].median())
    for c, mascara in faltaba.items():
        panel[c + "_imputado"] = mascara.astype(int)
        print(f"{c}: {int(mascara.sum()):,} valores imputados")
    panel["fase_enso"] = np.select([panel.ONI >= 0.5, panel.ONI <= -0.5], ["El Nino", "La Nina"], "Neutral")

    salida = {k: panel[panel._parte == k].sort_values("_i").drop(columns=["_parte", "_i"]).reset_index(drop=True)
              for k in partes}
    salida["train"]["tiene_evento"] = salida["train"]["tiene_evento"].astype(int)
    for k in ("pub", "priv"):
        salida[k] = salida[k].drop(columns=OBJETIVO + HISTORIAL, errors="ignore")
    return salida["train"], salida["pub"], salida["priv"]


train, test_pub, test_priv = preparar(train, test_pub, test_priv)
print(train.shape, test_pub.shape, test_priv.shape)

Objetivo corregido según la definición oficial: 0 filas
densidad_poblacional: 12 valores imputados
ingresos_tributarios: 36 valores imputados
inversion_gestion_riesgo: 96 valores imputados
nbi_pct: 96 valores imputados
(62814, 29) (20938, 17) (22040, 17)


## 2. Historial (solo pasado, jerárquico) y variables

In [4]:
K = {"dep": 120, "nm": 1102, "dm": 60, "mun": 24, "mm": 3, "mag": 3}
HIST_COLS = ["hist_tasa_mun", "hist_tasa_mes", "hist_tasa_dep", "hist_tasa_dep_mes", "hist_tasa_nac_mes",
             "hist_mag_mun", "hist_esperado_log", "hist_pct_inund", "hist_pct_masa", "hist_reciente_12"]


def _conteos(hist, claves, p):
    return (hist.groupby(claves).agg(**{f"e_{p}": ("tiene_evento", "sum"), f"n_{p}": ("tiene_evento", "size"),
                                         f"lp_{p}": ("logp", "sum")}).reset_index())


def historial_congelado(hist, filas):
    """Historial calculado con todo `hist` (que es pasado respecto a `filas`), aplicado a `filas`."""
    hist = hist.assign(logp=np.log1p(hist["personas_desplazadas"]), dep=hist["DIVIPOLA"].str[:2])
    f = filas.assign(dep=filas["DIVIPOLA"].str[:2])
    for claves, p in [(["DIVIPOLA"], "mun"), (["DIVIPOLA", "mes"], "mm"), (["dep"], "dep"),
                      (["dep", "mes"], "dm"), (["mes"], "nm")]:
        f = f.merge(_conteos(hist, claves, p), on=claves, how="left")
    c = f.fillna({col: 0 for col in f.columns if col[:2] in ("e_", "n_") or col.startswith("lp_")})
    tasa_nac = hist["tiene_evento"].mean()
    tasa_nm = (c.e_nm + K["nm"] * tasa_nac) / (c.n_nm + K["nm"])
    tasa_dep = (c.e_dep + K["dep"] * tasa_nac) / (c.n_dep + K["dep"])
    tasa_dm = (c.e_dm + K["dm"] * tasa_dep * tasa_nm / tasa_nac) / (c.n_dm + K["dm"])
    tasa_mun = (c.e_mun + K["mun"] * tasa_dep) / (c.n_mun + K["mun"])
    tasa_mm = (c.e_mm + K["mm"] * tasa_mun * tasa_dm / tasa_dep) / (c.n_mm + K["mm"])
    mag_nac = hist["logp"].sum() / max(hist["tiene_evento"].sum(), 1)
    mag_dep = (c.lp_dep + 5 * mag_nac) / (c.e_dep + 5)
    mag_mun = (c.lp_mun + K["mag"] * mag_dep) / (c.e_mun + K["mag"])

    tipos = hist.groupby("DIVIPOLA")[["n_inundacion", "n_movimiento_masa", "n_eventos"]].sum()
    reciente = hist[hist["fecha"] > hist["fecha"].max() - pd.DateOffset(months=12)].groupby("DIVIPOLA")["tiene_evento"].mean()
    salida = filas.copy()
    salida["hist_tasa_mun"] = tasa_mun.to_numpy()
    salida["hist_tasa_mes"] = tasa_mm.to_numpy()
    salida["hist_tasa_dep"] = tasa_dep.to_numpy()
    salida["hist_tasa_dep_mes"] = tasa_dm.to_numpy()
    salida["hist_tasa_nac_mes"] = tasa_nm.to_numpy()
    salida["hist_mag_mun"] = mag_mun.to_numpy()
    salida["hist_esperado_log"] = (tasa_mm * mag_mun).to_numpy()   # E[log1p(personas)] según la historia
    salida["hist_pct_inund"] = filas["DIVIPOLA"].map((tipos.n_inundacion + 1) / (tipos.n_eventos + 3)).fillna(1 / 3).to_numpy()
    salida["hist_pct_masa"] = filas["DIVIPOLA"].map((tipos.n_movimiento_masa + 1) / (tipos.n_eventos + 3)).fillna(1 / 3).to_numpy()
    salida["hist_reciente_12"] = filas["DIVIPOLA"].map(reciente).fillna(tasa_nac).to_numpy()
    # Nivel nacional de los últimos 12 meses de la historia (tendencia de reporte); solo lo usa la variante E3
    nivel_nac = hist.loc[hist["fecha"] > hist["fecha"].max() - pd.DateOffset(months=12), "tiene_evento"].mean()
    salida["hist_nivel_nac_12"] = nivel_nac
    salida["hist_tendencia_nac"] = nivel_nac / tasa_nac
    return salida


def historial_entrenamiento(hist):
    """Filas de entrenamiento: para cada año Y (desde el segundo), historial con los años anteriores a Y."""
    anios = sorted(hist["anio"].unique())
    return pd.concat([historial_congelado(hist[hist.anio < a], hist[hist.anio == a]) for a in anios[1:]],
                     ignore_index=True)

In [5]:
ONI_SERIE = pd.concat([train, test_pub, test_priv])[["fecha", "ONI"]].groupby("fecha")["ONI"].median()


NIVEL_COLS = ["hist_nivel_nac_12", "hist_tendencia_nac"]


def features(df, nivel_nacional=False):
    X = pd.DataFrame(index=df.index)
    for c in COV_ANUALES:
        X["log_" + c] = np.log1p(df[c].clip(lower=0))
    X["ingresos_rango_anio"] = df.groupby("anio")["ingresos_tributarios"].rank(pct=True)
    for c in COV_ESTATICAS + ["ONI"] + HIST_COLS:
        X[c] = df[c]
    for c in [c for c in df.columns if c.endswith("_imputado")]:
        X[c] = df[c]
    for k in (1, 2, 3):
        X[f"ONI_rezago_{k}"] = (df["fecha"] - pd.DateOffset(months=k)).map(ONI_SERIE).to_numpy()
    X["mes"] = df["mes"]
    X["mes_sin"], X["mes_cos"] = np.sin(2 * np.pi * df["mes"] / 12), np.cos(2 * np.pi * df["mes"] / 12)
    X["departamento"] = df["DIVIPOLA"].str[:2].astype(int)
    X["oni_x_inundacion"] = df["ONI"] * df["hist_pct_inund"]
    X["oni_x_tasa"] = df["ONI"] * df["hist_tasa_mun"]
    if nivel_nacional:
        for c in NIVEL_COLS:
            X[c] = df[c]
    return X

## 3. Métricas oficiales y validador de la plantilla

In [6]:
from sklearn.metrics import roc_auc_score

def metricas(pred, verdad):
    m = verdad.merge(pred, on=["DIVIPOLA", "fecha"])
    y = np.log1p(m["personas_desplazadas"])
    est = np.log1p(m["personas_desplazadas_estimadas"])
    lo, hi = np.log1p(m["personas_desplazadas_q10"]), np.log1p(m["personas_desplazadas_q90"])
    wink = (hi - lo) + 10 * np.clip(lo - y, 0, None) + 10 * np.clip(y - hi, 0, None)   # alpha = 0.2
    return {"auc": roc_auc_score(m["tiene_evento"], m["prob_evento"]),
            "rmse_log": float(np.sqrt(np.mean((y - est) ** 2))), "winkler": float(wink.mean())}

def puntos(met, nula):
    """Parte automática de la nota (máx. 75 puntos), como la tabla de posiciones."""
    s_auc = max(0, (met["auc"] - 0.5) / 0.5)
    s_rmse = min(1, max(0, 1 - met["rmse_log"] / nula["rmse_log"]))
    s_wink = min(1, max(0, 1 - met["winkler"] / nula["winkler"]))
    return 100 * (0.50 * s_auc + 0.20 * s_rmse + 0.05 * s_wink)


def prediccion_nula(filas):
    return filas[["DIVIPOLA", "fecha"]].assign(prob_evento=0.5, personas_desplazadas_estimadas=0.0,
                                               personas_desplazadas_q10=0.0, personas_desplazadas_q90=0.0)


def winkler_con_factor(pred, verdad, factor):
    m = verdad.merge(pred, on=["DIVIPOLA", "fecha"])
    y, lo = np.log1p(m["personas_desplazadas"]), np.log1p(m["personas_desplazadas_q10"])
    hi = np.maximum(lo, np.log1p(m["personas_desplazadas_q90"]) * factor)
    return float(((hi - lo) + 10 * np.clip(lo - y, 0, None) + 10 * np.clip(y - hi, 0, None)).mean())

In [7]:
COLUMNAS = ["DIVIPOLA", "fecha", "prob_evento", "personas_desplazadas_estimadas",
            "personas_desplazadas_q10", "personas_desplazadas_q90"]

def validar_entrega(e, test_pub=test_pub, test_priv=test_priv):
    errores = []
    faltan = [c for c in COLUMNAS if c not in e.columns]
    if faltan:
        return print("❌ Faltan columnas:", faltan)
    claves = pd.concat([test_pub, test_priv])[["DIVIPOLA", "fecha"]].assign(fecha=lambda d: pd.to_datetime(d.fecha))
    e2 = e.assign(DIVIPOLA=e.DIVIPOLA.astype(str).str.zfill(5), fecha=pd.to_datetime(e.fecha))
    if e2.duplicated(["DIVIPOLA", "fecha"]).any():
        errores.append("hay filas DIVIPOLA-fecha duplicadas")
    faltantes = len(claves.merge(e2, on=["DIVIPOLA", "fecha"], how="left", indicator=True).query("_merge == 'left_only'"))
    if faltantes:
        errores.append(f"faltan {faltantes} filas de las {len(claves)} esperadas")
    num = e2[COLUMNAS[2:]].apply(pd.to_numeric, errors="coerce")
    if num.isna().any().any():
        errores.append("hay valores vacíos o no numéricos")
    if not num.prob_evento.between(0, 1).all():
        errores.append("prob_evento debe estar entre 0 y 1")
    if (num[COLUMNAS[3:]] < 0).any().any():
        errores.append("las columnas de personas deben ser >= 0")
    if (num.personas_desplazadas_q10 > num.personas_desplazadas_q90).any():
        errores.append("personas_desplazadas_q10 debe ser <= personas_desplazadas_q90")
    print("✅ Entrega válida:", len(e), "filas" if not errores else "", *["\n❌ " + x for x in errores])
    return not errores

## 4. Variantes y entrenamiento
`CONFIG_BASE` es el control (`E1_arboles_400`). Cada variante cambia una sola cosa, salvo `COMBO`.

In [8]:
import shutil
import time

import lightgbm as lgb

NOMBRE_EQUIPO = "AICOLOMBIA"
SEMILLA = 42
LGBM_BASE = dict(learning_rate=0.05, num_leaves=31, min_child_samples=100, colsample_bytree=0.8, subsample=0.8,
                 subsample_freq=1, reg_lambda=1.0, random_state=SEMILLA, deterministic=True, force_row_wise=True,
                 verbose=-1)
ES_MESES, ES_PACIENCIA, FACTOR_ARBOLES, MAX_ARBOLES = 12, 100, 1.15, 2000

CONFIG_BASE = dict(early_stopping=False, arboles_p=400, arboles_mag=300, arboles_q=300,
                   hurdle=False, nivel_nacional=False, lgbm={})
VARIANTES = {
    "V1_early_stopping": dict(early_stopping=True),
    "E1_arboles_200": dict(arboles_p=200),
    "E1_arboles_400": dict(),
    "E1_arboles_800": dict(arboles_p=800),
    "E2_hurdle": dict(hurdle=True),
    "E3_nivel_nacional": dict(nivel_nacional=True),
    "E4_min_hoja_200": dict(lgbm=dict(min_child_samples=200)),
    "E4_hojas_15": dict(lgbm=dict(num_leaves=15)),
    "COMBO": dict(hurdle=True, nivel_nacional=True, lgbm=dict(min_child_samples=200)),
}
DESCRIPCION = {
    "V1_early_stopping": "Versión 1 (early stopping)",
    "E1_arboles_200": "Clasificador 200 árboles fijos",
    "E1_arboles_400": "Clasificador 400 árboles fijos (control)",
    "E1_arboles_800": "Clasificador 800 árboles fijos",
    "E2_hurdle": "Control + magnitud en dos partes",
    "E3_nivel_nacional": "Control + nivel nacional reciente",
    "E4_min_hoja_200": "Control + mínimo 200 filas por hoja",
    "E4_hojas_15": "Control + 15 hojas",
    "COMBO": "Control + dos partes + nivel nacional + 200 por hoja",
}


def ajustar(X, y, fechas, clase, cfg, n_arboles, **extra):
    params = {**LGBM_BASE, **cfg["lgbm"], **extra}
    if not cfg["early_stopping"]:
        return clase(**params, n_estimators=n_arboles).fit(X, y), n_arboles
    meses = np.sort(fechas.unique())
    corte = meses[-min(ES_MESES, max(3, len(meses) // 4))]
    a, b = (fechas < corte).to_numpy(), (fechas >= corte).to_numpy()
    m = clase(**params, n_estimators=MAX_ARBOLES).fit(X[a], y[a], eval_set=[(X[b], y[b])],
                                                     callbacks=[lgb.early_stopping(ES_PACIENCIA, verbose=False)])
    n = max(50, int(round((m.best_iteration_ or MAX_ARBOLES) * FACTOR_ARBOLES)))
    return clase(**params, n_estimators=n).fit(X, y), n


def entrenar(filas, cfg):
    X, fechas = features(filas, cfg["nivel_nacional"]), filas["fecha"]
    y_ev, y_log = filas["tiene_evento"], np.log1p(filas["personas_desplazadas"])
    modelos, arboles = {}, {}
    modelos["p"], arboles["p"] = ajustar(X, y_ev, fechas, lgb.LGBMClassifier, cfg, cfg["arboles_p"],
                                         objective="binary", metric="auc")
    filas_mag = (y_ev == 1).to_numpy() if cfg["hurdle"] else np.ones(len(filas), bool)
    modelos["e"], arboles["e"] = ajustar(X[filas_mag], y_log[filas_mag], fechas[filas_mag], lgb.LGBMRegressor, cfg,
                                         cfg["arboles_mag"], objective="regression", metric="l2")
    for q in (0.1, 0.9):
        modelos[q], arboles[q] = ajustar(X, y_log, fechas, lgb.LGBMRegressor, cfg, cfg["arboles_q"],
                                         objective="quantile", alpha=q, metric="quantile")
    return modelos, arboles


def predecir(modelos, filas, cfg, factor_q90=1.0):
    X = features(filas, cfg["nivel_nacional"])
    p = filas[["DIVIPOLA", "fecha"]].copy()
    prob = modelos["p"].predict_proba(X)[:, 1]
    p["prob_evento"] = prob
    magnitud = np.clip(modelos["e"].predict(X), 0, None)          # en escala log1p
    p["personas_desplazadas_estimadas"] = np.expm1(prob * magnitud if cfg["hurdle"] else magnitud)
    lo = np.clip(modelos[0.1].predict(X), 0, None)
    hi = np.clip(modelos[0.9].predict(X), 0, None) * factor_q90
    p["personas_desplazadas_q10"] = np.expm1(np.minimum(lo, hi))
    p["personas_desplazadas_q90"] = np.expm1(np.maximum(lo, hi))
    return p

## 5. Correr todas las variantes
El historial de cada corte se calcula una sola vez y se reutiliza. Por cada variante: validación en F1, F2 y F3, factor de
`q90` que minimiza el Winkler, modelo final con todo el entrenamiento y CSV de entrega validado en `entregas/`.

In [9]:
CORTES = [("F1", "2020-09-01", "2020-10-01", "2021-09-01"),
          ("F2", "2021-09-01", "2021-10-01", "2022-09-01"),
          ("F3", "2020-09-01", "2021-10-01", "2022-09-01")]
FACTORES = [0.7, 0.8, 0.9, 1.0, 1.1, 1.25, 1.4]

t0 = time.time()
PREP = {}
for nombre, fin_hist, ini_val, fin_val in CORTES:
    hist = train[train.fecha <= fin_hist]
    va = train[(train.fecha >= ini_val) & (train.fecha <= fin_val)]
    PREP[nombre] = (historial_entrenamiento(hist), historial_congelado(hist, va), va)
PRUEBA = pd.concat([test_pub, test_priv], ignore_index=True)
FINAL = (historial_entrenamiento(train), historial_congelado(train, PRUEBA))
print(f"Historial preparado en {time.time() - t0:.0f}s")

os.makedirs("entregas", exist_ok=True)
resumen, detalle = [], []
for variante, cambios in VARIANTES.items():
    t1 = time.time()
    cfg = {**CONFIG_BASE, **cambios}
    por_corte, factores = {}, []
    for corte, (filas_hist, filas_va, va) in PREP.items():
        modelos_cv, _ = entrenar(filas_hist, cfg)
        pred = predecir(modelos_cv, filas_va, cfg)
        met, nula = metricas(pred, va), metricas(prediccion_nula(va), va)
        winklers = {f: winkler_con_factor(pred, va, f) for f in FACTORES}
        factores.append(min(winklers, key=winklers.get))
        m = va.merge(pred, on=["DIVIPOLA", "fecha"])
        cobertura = float(((m.personas_desplazadas >= m.personas_desplazadas_q10) &
                           (m.personas_desplazadas <= m.personas_desplazadas_q90)).mean())
        por_corte[corte] = {**met, "puntos": puntos(met, nula), "cobertura": cobertura}
        detalle.append({"variante": variante, "corte": corte, **por_corte[corte]})
    factor = float(np.median(factores))

    modelos_fin, arboles_fin = entrenar(FINAL[0], cfg)
    entrega = predecir(modelos_fin, FINAL[1], cfg, factor)
    entrega["fecha"] = entrega["fecha"].dt.strftime("%Y-%m-%d")
    entrega = entrega[COLUMNAS]
    valida = bool(validar_entrega(entrega))
    archivo = f"entregas/entrega_{NOMBRE_EQUIPO}_{variante}.csv"
    if valida:
        entrega.to_csv(archivo, index=False)

    d = pd.DataFrame(por_corte).T
    resumen.append({"variante": variante, "descripcion": DESCRIPCION[variante],
                    "puntos_prom": d.puntos.mean(), "puntos_desv": d.puntos.std(),
                    "puntos_F1": d.loc["F1", "puntos"], "puntos_F2": d.loc["F2", "puntos"],
                    "puntos_F3": d.loc["F3", "puntos"], "auc_prom": d.auc.mean(), "rmse_log_prom": d.rmse_log.mean(),
                    "winkler_prom": d.winkler.mean(), "cobertura_prom": d.cobertura.mean(), "factor_q90": factor,
                    "arboles_final": arboles_fin, "archivo": archivo if valida else "INVÁLIDA",
                    "segundos": round(time.time() - t1)})
    print(f"{variante:20s} puntos {d.puntos.mean():5.1f} ± {d.puntos.std():.1f} · F3 {d.loc['F3', 'puntos']:5.1f} "
          f"· {time.time() - t1:.0f}s")
print(f"Total: {(time.time() - t0) / 60:.1f} min")

Historial preparado en 2s


c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

✅ Entrega válida: 42978 filas
V1_early_stopping    puntos  20.1 ± 3.9 · F3  16.2 · 18s
✅ Entrega válida: 42978 filas
E1_arboles_200       puntos  19.7 ± 3.4 · F3  16.0 · 12s
✅ Entrega válida: 42978 filas
E1_arboles_400       puntos  19.3 ± 3.2 · F3  16.0 · 14s
✅ Entrega válida: 42978 filas
E1_arboles_800       puntos  18.7 ± 3.2 · F3  15.3 · 19s
✅ Entrega válida: 42978 filas
E2_hurdle            puntos  19.3 ± 3.2 · F3  15.9 · 13s
✅ Entrega válida: 42978 filas
E3_nivel_nacional    puntos  19.5 ± 3.1 · F3  16.3 · 16s
✅ Entrega válida: 42978 filas
E4_min_hoja_200      puntos  19.6 ± 3.4 · F3  16.0 · 15s
✅ Entrega válida: 42978 filas
E4_hojas_15          puntos  19.7 ± 3.5 · F3  16.1 · 12s
✅ Entrega válida: 42978 filas
COMBO                puntos  19.5 ± 3.1 · F3  16.4 · 13s
Total: 2.2 min


## 6. Resultados

In [10]:
res = pd.DataFrame(resumen)
v1 = res.set_index("variante").loc["V1_early_stopping"]
res["delta_vs_V1"] = res["puntos_prom"] - v1["puntos_prom"]
res["cumple_regla"] = (res["delta_vs_V1"] >= 0.5) & (res["puntos_F3"] >= v1["puntos_F3"])
tabla = res.sort_values("puntos_prom", ascending=False).reset_index(drop=True)
columnas = ["variante", "descripcion", "puntos_prom", "puntos_desv", "puntos_F1", "puntos_F2", "puntos_F3",
            "delta_vs_V1", "cumple_regla", "auc_prom", "rmse_log_prom", "winkler_prom", "cobertura_prom",
            "factor_q90", "archivo"]
pd.set_option("display.width", 250)
display(tabla[columnas].round(3))

print("Detalle por corte:")
display(pd.DataFrame(detalle).pivot(index="variante", columns="corte",
                                    values=["auc", "rmse_log", "winkler", "puntos"]).round(3))

tabla.to_csv("entregas/resultados_experimentos.csv", index=False)
candidatas = tabla[tabla["cumple_regla"]]
if len(candidatas):
    mejor = candidatas.iloc[0]
    print(f"Recomendada para entregar: {mejor.variante} ({mejor.descripcion}) · "
          f"{mejor.puntos_prom:.1f} puntos (+{mejor.delta_vs_V1:.1f} vs V1) · archivo {mejor.archivo}")
else:
    print("Ninguna variante cumple la regla (≥ 0.5 puntos sobre V1 sin empeorar F3): mantener la entrega V1.")

,variante,descripcion,puntos_prom,puntos_desv,puntos_F1,puntos_F2,puntos_F3,delta_vs_V1,cumple_regla,auc_prom,rmse_log_prom,winkler_prom,cobertura_prom,factor_q90,archivo
0,V1_early_stopping,Versión 1 (early stopping),20.064,3.913,19.872,24.070,16.250,0.000,False,0.683,1.295,3.414,0.918,1.0,entregas/entrega_AICOLOMBIA_V1_early_stopping.csv
1,E4_hojas_15,Control + 15 hojas,19.734,3.497,19.922,23.134,16.147,-0.329,False,0.681,1.306,3.414,0.920,1.0,entregas/entrega_AICOLOMBIA_E4_hojas_15.csv
2,E1_arboles_200,Clasificador 200 árboles fijos,19.681,3.430,20.163,22.843,16.035,-0.383,False,0.681,1.311,3.422,0.919,1.0,entregas/entrega_AICOLOMBIA_E1_arboles_200.csv
3,E4_min_hoja_200,Control + mínimo 200 filas por hoja,19.584,3.415,19.873,22.845,16.034,-0.480,False,0.680,1.307,3.405,0.919,1.0,entregas/entrega_AICOLOMBIA_E4_min_hoja_200.csv
4,COMBO,Control + dos partes + nivel nacional + 200 po...,19.539,3.134,19.621,22.632,16.364,-0.525,False,0.680,1.312,3.395,0.921,1.0,entregas/entrega_AICOLOMBIA_COMBO.csv
5,E3_nivel_nacional,Control + nivel nacional reciente,19.523,3.133,19.799,22.510,16.261,-0.540,False,0.680,1.313,3.404,0.918,1.0,entregas/entrega_AICOLOMBIA_E3_nivel_nacional.csv
6,E1_arboles_400,Clasificador 400 árboles fijos (control),19.347,3.176,19.715,22.324,16.004,-0.716,False,0.678,1.311,3.422,0.919,1.0,entregas/entrega_AICOLOMBIA_E1_arboles_400.csv
7,E2_hurdle,Control + magnitud en dos partes,19.300,3.250,19.542,22.422,15.935,-0.764,False,0.678,1.314,3.422,0.919,1.0,entregas/entrega_AICOLOMBIA_E2_hurdle.csv
8,E1_arboles_800,Clasificador 800 árboles fijos,18.742,3.221,19.142,21.743,15.339,-1.322,False,0.672,1.311,3.422,0.919,1.0,entregas/entrega_AICOLOMBIA_E1_arboles_800.csv


Detalle por corte:


auc               rmse_log               winkler                puntos                
corte                 F1     F2     F3       F1     F2     F3      F1     F2     F3      F1      F2      F3
variante                                                                                                   
COMBO              0.679  0.709  0.652    1.375  1.269  1.292   3.452  3.288  3.446  19.621  22.632  16.364
E1_arboles_200     0.683  0.712  0.648    1.364  1.279  1.289   3.509  3.269  3.487  20.163  22.843  16.035
E1_arboles_400     0.679  0.707  0.648    1.364  1.279  1.289   3.509  3.269  3.487  19.715  22.324  16.004
E1_arboles_800     0.673  0.701  0.642    1.364  1.279  1.289   3.509  3.269  3.487  19.142  21.743  15.339
E2_hurdle          0.679  0.707  0.648    1.376  1.272  1.294   3.509  3.269  3.487  19.542  22.422  15.935
E3_nivel_nacional  0.680  0.709  0.651    1.366  1.280  1.294   3.479  3.271  3.464  19.799  22.510  16.261
E4_hojas_15        0.681  0.714  0.649    1.361  1.271  1.288   3.501  3.296  3.446  19.922  23.134  16.147
E4_min_hoja_200    0.679  0.712  0.648    1.359  1.274  1.287   3.459  3.287  3.469  19.873  22.845  16.034
V1_early_stopping  0.679  0.722  0.647    1.353  1.263  1.268   3.513  3.248  3.482  19.872  24.070  16.250

Ninguna variante cumple la regla (≥ 0.5 puntos sobre V1 sin empeorar F3): mantener la entrega V1.


## 7. Descargar los CSV
Un zip con un CSV por variante y la tabla de resultados. Para la tabla en vivo, suban **solo la recomendada** (máximo
una entrega cada 30 minutos; cuenta la última válida) y anoten el resultado en la bitácora.

In [11]:
shutil.make_archive("entregas_experimentos", "zip", "entregas")
try:
    from google.colab import files
    files.download("entregas_experimentos.zip")
except ImportError:
    print("Guardado en", os.path.abspath("entregas_experimentos.zip"))

Guardado en g:\My Drive\Documentos Juan Felipe\Hackaton_PENUD\notebooks\entregas_experimentos.zip
